In [ ]:
%load_ext autoreload
%autoreload 2
import sys

sys.path.insert(0, "../")

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import fasttext
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

# Set plotting style
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# Load Env
load_dotenv("../.env")

In [ ]:
# Load the dataset
# Hand-labeled workbook incl. eHRAF text (not distributed; see README)
data_path = Path("../data/intermediate/TheTruthV7.xlsx")
df = pd.read_excel(data_path).dropna(subset=["Text"]).reset_index(drop=True)
print(df.shape)
df.head(3)

# **Language Identification with GlotLID**

In [ ]:
cache_dir = Path("../data/intermediate/models")
cache_dir.mkdir(parents=True, exist_ok=True)

print("Downloading GlotLID model (this may take a few minutes)...")
model_path = hf_hub_download(
    repo_id="cis-lmu/glotlid", filename="model.bin", cache_dir=str(cache_dir)
)

print(f"Model downloaded to: {model_path}")
model = fasttext.load_model(model_path)
# labels = model.get_labels()

print("Model loaded successfully!")
print(f"Model supports {len(model.labels)} language labels")

In [ ]:
def identify_language(text, model, k=3):
    """
    Identify language of text using GlotLID model.

    Args:
        text (str): Input text string to identify.
        model (fasttext.FastText._FastText): Loaded fasttext GlotLID model.
        k (int): Number of top predictions to return.

    Returns:
        dict: Dictionary with keys 'lang_1', 'conf_1', ..., 'lang_k', 'conf_k'
            containing language predictions and confidence scores (None if unavailable).
    """
    if pd.isna(text) or str(text).strip() == "":
        return {
            "lang_1": None,
            "conf_1": None,
            "lang_2": None,
            "conf_2": None,
            "lang_3": None,
            "conf_3": None,
        }

    try:
        # Predict language
        labels, probs = model.predict(str(text), k=k)

        # Clean up labels (remove __label__ prefix)
        clean_labels = [label.replace("__label__", "") for label in labels]

        result = {}
        for i in range(k):
            if i < len(clean_labels):
                result[f"lang_{i + 1}"] = clean_labels[i]
                result[f"conf_{i + 1}"] = float(probs[i])
            else:
                result[f"lang_{i + 1}"] = None
                result[f"conf_{i + 1}"] = None

        return result

    except Exception as e:
        print(f"Error processing text: {e}")
        return {
            "lang_1": None,
            "conf_1": None,
            "lang_2": None,
            "conf_2": None,
            "lang_3": None,
            "conf_3": None,
        }

In [ ]:
# Test the function on a few examples
for idx, text in df[df["Text"].notna()]["Text"].head(3).items():
    text_preview = str(text)[:100] + "..." if len(str(text)) > 100 else str(text)
    result = identify_language(text, model, k=3)

    print(f"\nText sample: {text_preview}")
    print(f"  → Primary: {result['lang_1']} (confidence: {result['conf_1']:.4f})")
    print(f"  → Secondary: {result['lang_2']} (confidence: {result['conf_2']:.4f})")
    print(f"  → Tertiary: {result['lang_3']} (confidence: {result['conf_3']:.4f})")

In [ ]:
# Apply language identification to all texts using a straightforward for loop
lang_results = []
for text in tqdm(df["Text"], desc="Language Identification Progress"):
    result = identify_language(text, model, k=3)
    if not result:
        print("aaa")
    lang_results.append(result)

lang_df = pd.DataFrame(lang_results)
df_with_langs = pd.concat([df, lang_df], axis=1)

## Analysis of Language Distribution

In [ ]:
# Summary statistics
print("=== LANGUAGE IDENTIFICATION SUMMARY ===\n")

# Primary language distribution
primary_lang_counts = df_with_langs["lang_1"].value_counts()
print(f"Number of unique primary languages detected: {len(primary_lang_counts)}")
print("\nTop 10 primary languages:")
print(primary_lang_counts.head(10).to_string())

# Confidence statistics
print("\n=== CONFIDENCE STATISTICS ===")
print(f"Mean confidence (primary): {df_with_langs['conf_1'].mean():.4f}")
print(f"Median confidence (primary): {df_with_langs['conf_1'].median():.4f}")
print(f"Min confidence (primary): {df_with_langs['conf_1'].min():.4f}")
print(f"Max confidence (primary): {df_with_langs['conf_1'].max():.4f}")

# High confidence predictions
high_conf = df_with_langs[df_with_langs["conf_1"] > 0.9]
print(
    f"\nPredictions with >90% confidence: {len(high_conf)} ({len(high_conf) / len(df_with_langs) * 100:.1f}%)"
)

In [ ]:
languages = [
    "English",
    "French",
    "Southern Kalinga",
    "Balangao",
    "Amganad Ifugao",
    "Polish",
    "Korean",
    "Dutch",
    "Standard Malay",
    "Rarotongan (Cook Islands Māori)",
]
print(languages)

In [ ]:
# Confidence interval of entries where lang_1 is not 'eng_Latn'
non_eng = df_with_langs.loc[
    df_with_langs["lang_1"] != "eng_Latn", ["Text", "lang_1", "conf_1"]
].head(7)
for idx, row in non_eng.iterrows():
    print(f"Text: {row['Text']}\n[Language: {row['lang_1']} | Confidence: {row['conf_1']:.4f}]\n")

In [ ]:
eng_mask = df_with_langs["lang_1"] == "eng_Latn"
non_eng_mask = df_with_langs["lang_1"] != "eng_Latn"

conf_eng = df_with_langs.loc[eng_mask, "conf_1"].dropna()
conf_non_eng = df_with_langs.loc[non_eng_mask, "conf_1"].dropna()

fig, ax = plt.subplots(figsize=(8, 5))
sns.kdeplot(
    data=conf_eng, label=f"English n={len(conf_eng)}", fill=True, alpha=0.5, color="skyblue", ax=ax
)
sns.kdeplot(
    data=conf_non_eng,
    label=f"Non-English n={len(conf_non_eng)}",
    fill=True,
    alpha=0.5,
    color="salmon",
    ax=ax,
)

ax.set_xlabel("Confidence Score", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title(
    "Confidence Score Distribution: English vs Non-English", fontsize=14, fontweight="bold"
)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# NOTE: CLip distribution function.

In [ ]:
# Display sample results
print("=== SAMPLE RESULTS ===\n")
sample_cols = ["Text", "lang_1", "conf_1", "lang_2", "conf_2", "lang_3", "conf_3"]
available_cols = [col for col in sample_cols if col in df_with_langs.columns]

print("High confidence predictions (>95%):")
high_conf_sample = df_with_langs[df_with_langs["conf_1"] > 0.95][available_cols].head(5)
for idx, row in high_conf_sample.iterrows():
    print(f"[{idx}] {row['Text']!s}")
    print(f"    {row['lang_1']} ({row['conf_1']:.4f})")

print("\n\nLower confidence predictions (<70%):")
low_conf_sample = df_with_langs[df_with_langs["conf_1"] < 0.7][available_cols].head(5)
for idx, row in low_conf_sample.iterrows():
    print(f"[{idx}] {row['Text']!s}")
    print(
        f"    Top 3: {row['lang_1']} ({row['conf_1']:.4f}), {row['lang_2']} ({row['conf_2']:.4f}), {row['lang_3']} ({row['conf_3']:.4f})"
    )

# **Language Translation**

In [ ]:
import deepl
import pycountry
import torch
from transformers import AutoModelForSeq2SeqLM, NllbTokenizer

In [ ]:
def create_glotlid_deepl_mapping(deepl_langs):
    """
    Automatically create GlotLID → DeepL language mapping
    Uses pycountry library to convert ISO 639-1 → ISO 639-3

    Returns: dict mapping GlotLID codes (e.g., 'fra_Latn') to DeepL codes (e.g., 'FR')
    """
    # Create set of DeepL codes (2-letter base)
    deepl_codes = {lang.code.split("-")[0].upper() for lang in deepl_langs}

    # Build mapping
    mapping = {}

    for deepl_code in deepl_codes:
        try:
            # Convert DeepL code (ISO 639-1) to ISO 639-3 using pycountry
            lang = pycountry.languages.get(alpha_2=deepl_code.lower())
            if not lang:
                continue

            iso3 = lang.alpha_3  # ISO 639-3 code (e.g., 'fra', 'deu', 'eng')

            # Map common scripts for this language
            scripts = ["Latn"]  # Default: Latin script

            # Add Cyrillic for relevant languages
            if iso3 in ["rus", "bul", "ukr", "srp", "mkd", "bel"]:
                scripts.append("Cyrl")

            # Add Arabic script for relevant languages
            if iso3 in ["ara", "arb", "fas", "urd", "pus"]:
                scripts.append("Arab")

            # Greek script
            if iso3 == "ell":
                scripts = ["Grek"]

            # Japanese scripts
            if iso3 == "jpn":
                scripts = ["Jpan"]

            # Korean script
            if iso3 == "kor":
                scripts = ["Hang"]

            # Chinese scripts
            if iso3 in ["zho", "cmn"]:
                scripts = ["Hans", "Hant"]

            # Create mappings for each script
            for script in scripts:
                glotlid_code = f"{iso3}_{script}"
                mapping[glotlid_code] = deepl_code

        except Exception as e:
            # Language not found in pycountry
            print(f"⚠️ Could not map DeepL code '{deepl_code}': {e}")
            continue

    return mapping

In [ ]:
DEEPL_API_KEY = os.environ.get("DEEPL_API_KEY")
deepl_translator = deepl.Translator(DEEPL_API_KEY)
languages = deepl_translator.get_target_languages()

glotlid_deepl_lang_id_map = create_glotlid_deepl_mapping(languages)

usage = deepl_translator.get_usage()
if usage.character.limit_exceeded:
    print("⚠️ DeepL character limit exceeded!")
else:
    print(f"DeepL usage: {usage.character.count:,} / {usage.character.limit:,} characters")

In [ ]:
# Load NLLB-200 model for fallback translation
# NOTE: This is a large model (~1.3GB). First download may take time.
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    model_name = "facebook/nllb-200-distilled-600M"
    nllb_tokenizer = NllbTokenizer.from_pretrained(model_name, cache_dir="../data/models")
    nllb_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, cache_dir="../data/models")

    # Move to GPU if available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    nllb_model = nllb_model.to(device)

except Exception as e:
    print(f"❌ Failed to load NLLB-200: {e}")
    print("Translation will be limited to DeepL-supported languages only")
    nllb_model = None
    nllb_tokenizer = None
    device = "cpu"

In [ ]:
def translate_with_deepl(text, source_lang_code):
    """
    Translate text using DeepL API

    Args:
        text: Text to translate
        source_lang_code: GlotLID language code (e.g., 'fra_Latn')

    Returns:
        Translated text or None if failed
    """
    if not deepl_translator:
        return None

    if source_lang_code not in glotlid_deepl_lang_id_map:
        return None

    try:
        deepl_lang = glotlid_deepl_lang_id_map[source_lang_code]
        result = deepl_translator.translate_text(text, source_lang=deepl_lang, target_lang="EN-US")
        return result.text
    except Exception as e:
        print(f"DeepL translation failed: {e}")
        return None


def translate_with_nllb(text, source_lang_code):
    """
    Translate text using NLLB-200 model

    Args:
        text: Text to translate
        source_lang_code: GlotLID language code (e.g., 'fra_Latn')

    Returns:
        Translated text or None if failed
    """
    if not nllb_model or not nllb_tokenizer:
        return None

    try:
        # Set source language
        nllb_tokenizer.src_lang = source_lang_code

        # Tokenize
        inputs = nllb_tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Generate translation to English
        with torch.no_grad():
            outputs = nllb_model.generate(
                **inputs,
                forced_bos_token_id=nllb_tokenizer.lang_code_to_id.get("eng_Latn", 256047),
                max_length=512,
                num_beams=4,
                early_stopping=True,
            )

        # Decode
        translation = nllb_tokenizer.decode(outputs[0], skip_special_tokens=True)
        return translation

    except Exception as e:
        print(f"NLLB-200 translation failed for {source_lang_code}: {e}")
        return None


def translate_to_english(text, detected_lang):
    """
    Smart translation router: Try DeepL first, fall back to NLLB-200

    Args:
        text: Text to translate
        detected_lang: GlotLID language code

    Returns:
        dict with translation info
    """
    # Already English
    if detected_lang == "eng_Latn" or pd.isna(detected_lang):
        return {
            "translated_text": text,
            "translation_method": "original_english",
            "translation_success": True,
        }

    # Try DeepL first (if language is supported)
    if detected_lang in glotlid_deepl_lang_id_map:
        translated = translate_with_deepl(text, detected_lang)
        if translated:
            return {
                "translated_text": translated,
                "translation_method": "deepl",
                "translation_success": True,
            }

    # Fall back to NLLB-200
    translated = translate_with_nllb(text, detected_lang)
    if translated:
        return {
            "translated_text": translated,
            "translation_method": "nllb200",
            "translation_success": True,
        }

    # Translation failed
    return {
        "translated_text": text,
        "translation_method": "none_failed",
        "translation_success": False,
    }

In [ ]:
# Identify which texts need translation
non_english_mask = df_with_langs["lang_1"] != "eng_Latn"
texts_to_translate = df_with_langs[non_english_mask]

print(f"Total texts: {len(df_with_langs)}")
print(f"English texts: {(~non_english_mask).sum()}")
print(f"Non-English texts to translate: {len(texts_to_translate)}")

In [ ]:
# Translate all non-English texts
translation_results = []

for idx, row in df_with_langs.iterrows():
    text = row["Text"]
    lang = row["lang_1"]

    # Translate
    result = translate_to_english(text, lang)
    translation_results.append(result)

    # Print progress for non-English translations
    if result["translation_method"] not in ["original_english", "none_failed"]:
        print(f"[{idx}] {lang} → {result['translation_method']}")
        print(f"  Original: {str(text)[:100]}...")
        print(f"  Translated: {result['translated_text'][:100]}...")
        print()

print("=" * 60)
print("Translation complete!")

# Add translation results to dataframe
translation_df = pd.DataFrame(translation_results)
df_with_translations = pd.concat([df_with_langs.reset_index(drop=True), translation_df], axis=1)

print("\nTranslation summary:")
print(translation_df["translation_method"].value_counts())
print(
    f"\nSuccess rate: {translation_df['translation_success'].sum() / len(translation_df) * 100:.1f}%"
)

In [ ]:
# ksc_Latn - Southern Kalinga (Philippine language)
# blw_Latn - Balangao (Philippine language)
# ifa_Latn - Amganad Ifugao (Philippine language)

In [ ]:
# Display translation examples
print("=== TRANSLATION EXAMPLES ===\n")

# Show non-English translations
non_english_trans = df_with_translations[
    df_with_translations["translation_method"].isin(["deepl", "glot500"])
]

if len(non_english_trans) > 0:
    print(f"Showing {min(5, len(non_english_trans))} translation examples:\n")

    for idx, row in non_english_trans.head(5).iterrows():
        print(f"[{idx}] Language: {row['lang_1']} (confidence: {row['conf_1']:.3f})")
        print(f"Method: {row['translation_method']}")
        print(f"Original: {str(row['Text'])[:150]}...")
        print(f"Translated: {row['translated_text'][:150]}...")
        print("-" * 60)
        print()
else:
    print("No non-English texts were translated (all texts are English)")